Libraries and functions requiered for valdiation

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from UNet import UNet  # Look UNet.py
from dataloader import SolarPanelSegmentationDataset  #Look datalaoder code

Transformation, loading data and batch size definition

In [ ]:
# Define transformation for preprocessing
transform = transforms.Compose([
    transforms.ToTensor()
])

# Define dataset directories for validation
val_image_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\val_data\images'
val_mask_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\val_data\masks'

# Create a validation dataset
val_dataset = SolarPanelSegmentationDataset(val_image_dir, val_mask_dir, transform=transform)
val_dataloader = DataLoader(val_dataset, batch_size=40, shuffle=False)

# Initialize U-Net model
in_channels = 3  # RGB images
out_channels = 1  # Binary segmentation (solar panels or background)
model = UNet(in_channels, out_channels)

Load training model and define validation function

In [ ]:
model_path = r'C:\Users\32esla1mpg\Documents\UNet_PV\outputs\trained_PV.pth'
model.load_state_dict(torch.load(model_path))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Validation function
def validate_model(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    total_iou, total_dice, total_accuracy = 0.0, 0.0, 0.0
    with torch.no_grad():  # No need to track gradients
        for inputs, masks, _ in dataloader:
            inputs = inputs.to(device)
            masks = masks.to(device)
            outputs = model(inputs)

            # Convert outputs to binary predictions
            predictions = (outputs > 0.5).float()

            # Calculate metrics
            intersection = torch.logical_and(predictions, masks).sum().float()
            union = torch.logical_or(predictions, masks).sum().float()
            iou = intersection / (union + 1e-6)
            dice = (2 * intersection) / (predictions.sum().float() + masks.sum().float() + 1e-6)
            accuracy = (predictions == masks).sum().float() / torch.numel(masks)

            total_iou += iou
            total_dice += dice
            total_accuracy += accuracy

    # Average metrics over all batches in the dataloader
    average_iou = total_iou / len(dataloader)
    average_dice = total_dice / len(dataloader)
    average_accuracy = total_accuracy / len(dataloader)
    return average_accuracy, average_iou, average_dice

# Run validation
val_accuracy, val_iou, val_dice = validate_model(model, val_dataloader, device)
print(f'Validation Accuracy: {val_accuracy:.4f}, IoU: {val_iou:.4f}, Dice: {val_dice:.4f}')